<a href="https://colab.research.google.com/github/abhi1628/hybridgbrf/blob/main/HybridGBRF_Tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# hybridgbrf v1.0.0 Tutorial
## Gradient Boosted Random Forest - A Novel Hybrid Algorithm

This tutorial demonstrates hybridgbrf v1.0.0 with the actual hybrid algorithm:
- **Random Forest ensemble per boosting iteration** (variance reduction)
- **Sequential residual learning** (bias reduction)
- **Full sklearn compatibility** (pipelines, GridSearchCV, cross-validation)
- **Both regression and classification support**
- **Built-in early stopping**
- **Headless-safe plotting**

In [ ]:
!pip install hybridgbrf

---
## 1. Regression Example - California Housing

`fit(X, y)` trains on ALL provided data (sklearn standard).
User handles train/test split BEFORE calling fit.

In [ ]:
from hybridgbrf import GRFRegressor
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Load data
X, y = fetch_california_housing(return_X_y=True)

# USER handles splitting (sklearn standard)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize hybridgbrf with the novel hybrid parameters
model = GRFRegressor(
    n_iterations=50,              # Boosting rounds
    n_estimators_per_iteration=10, # Trees in each RF per round
    learning_rate=0.1,            # Shrinkage
    max_depth=3,                  # Tree depth
    subsample=0.8,                # Stochastic gradient boosting
    early_stopping_rounds=10,     # Stop if no improvement
    validation_fraction=0.1,    # Validation for early stopping
    random_state=42,
    verbose=1
)

# Fit on training data only (sklearn standard)
model.fit(X_train, y_train)

# Predict on test set
y_pred = model.predict(X_test)

# Evaluate (pass X AND y - sklearn standard)
print(f"R^2 Score: {model.score(X_test, y_test):.4f}")
print(f"MSE: {model.mse(X_test, y_test):.4f}")
print(f"MAE: {model.mae(X_test, y_test):.4f}")

# Check how many iterations actually ran (early stopping may have stopped early)
print(f"Iterations used: {len(model.estimators_)} / {model.n_iterations}")

---
## 2. Classification Example - Breast Cancer

Full classification support with `predict_proba()`

In [ ]:
from hybridgbrf import GRFClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load data
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Initialize classifier
clf = GRFClassifier(
    n_iterations=50,
    n_estimators_per_iteration=10,
    learning_rate=0.1,
    max_depth=3,
    early_stopping_rounds=10,
    random_state=42,
    verbose=1
)

# Fit
clf.fit(X_train, y_train)

# Predict classes
y_pred = clf.predict(X_test)

# Predict probabilities
y_proba = clf.predict_proba(X_test)

# Evaluate
print(f"Accuracy: {clf.score(X_test, y_test):.4f}")
print("\nClassification Report:")
print(clf.classification_report(X_test, y_test))
print("\nConfusion Matrix:")
print(clf.confusion_matrix(X_test, y_test))

---
## 3. Feature Importance & Training History Visualization

Works in headless environments (Colab, servers, CI).
Use `save_path` to save plots instead of displaying.

In [ ]:
# Plot aggregated feature importances across all boosting iterations
model.plot_feature_importances(
    top_n=15,
    save_path="/content/feature_importance.png",
    figsize=(10, 8)
)

# Plot training history with early stopping marker
model.plot_training_history(
    save_path="/content/training_history.png",
    figsize=(10, 6)
)

from IPython.display import Image, display
print("Feature Importances:")
display(Image("/content/feature_importance.png"))
print("\nTraining History:")
display(Image("/content/training_history.png"))

---
## 4. Cross-Validation

Full `cross_val_score` compatibility

In [ ]:
from sklearn.model_selection import cross_val_score

# Works with sklearn cross_val_score
scores = cross_val_score(
    model, X, y,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

print(f"CV MSE scores: {-scores}")
print(f"Mean CV MSE: {-scores.mean():.4f} (+/- {scores.std():.4f})")

---
## 5. Grid Search with sklearn

Works inside sklearn `GridSearchCV`

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_iterations': [30, 50, 100],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth': [2, 3, 5],
    'n_estimators_per_iteration': [5, 10, 20]
}

grid = GridSearchCV(
    GRFRegressor(random_state=42, verbose=0),
    param_grid,
    cv=3,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

print(f"Best params: {grid.best_params_}")
print(f"Best CV score: {-grid.best_score_:.4f}")

# Best model is already fitted
best_model = grid.best_estimator_
print(f"Test R^2: {best_model.score(X_test, y_test):.4f}")

---
## 6. Using in sklearn Pipeline

Full `Pipeline` compatibility

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('gbrf', GRFRegressor(
        n_iterations=50,
        n_estimators_per_iteration=10,
        learning_rate=0.1,
        random_state=42
    ))
])

pipeline.fit(X_train, y_train)
print(f"Pipeline R^2: {pipeline.score(X_test, y_test):.4f}")

---
## 7. Save & Load Model

`save_model()` and `load_model()` properly handle all state.

In [ ]:
# Save (built-in method)
model.save_model("/content/hybridgbrf_model.pkl")

# Load (class method)
from hybridgbrf import GRFRegressor
loaded_model = GRFRegressor.load_model("/content/hybridgbrf_model.pkl")

# Verify it works
print(f"Loaded model R^2: {loaded_model.score(X_test, y_test):.4f}")
print(f"Loaded model iterations: {len(loaded_model.estimators_)}")

---
## 8. The Hybrid Algorithm Explained

```
Standard Gradient Boosting:          hybridgbrf (Our Hybrid):
+--------------------------------+   +--------------------------------+
|  Iteration 1                   |   |  Iteration 1                   |
|  Single Tree -> Residuals      |   |  RF Ensemble -> Residuals      |
+--------------------------------+   +--------------------------------+
              |                                    |
              v                                    v
+--------------------------------+   +--------------------------------+
|  Iteration 2                   |   |  Iteration 2                   |
|  Single Tree -> Residuals      |   |  RF Ensemble -> Residuals      |
+--------------------------------+   +--------------------------------+
              |                                    |
              v                                    v
       ... continues ...                    ... continues ...

Key Difference: Single Tree vs Random Forest per iteration
Result: Better variance control + bias reduction
```